In [ ]:
%pip install pandas==2.2.3
%pip install scikit-learn==1.6.0
%pip install matplotlib==3.9.3
%pip install kagglehub[pandas-datasets]

In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

In [ ]:
import kagglehub
from pathlib import Path

download_dir = Path.cwd() / "data"
download_dir.mkdir(exist_ok=True)

dataset_file = download_dir / "AIML Dataset.csv"

# Download only if the file doesn't already exist
if not dataset_file.exists():
    print("Dataset not found. Downloading...")
    
    path = kagglehub.dataset_download(
        "amanalisiddiqui/fraud-detection-dataset",
        output_dir=str(download_dir)
    )
    
    print(f"Downloaded to: {path}")
else:
    print(f"Dataset already exists: {dataset_file}")

# Use the file
print(dataset_file)

In [ ]:
csv_file = ".\data\AIML Dataset.csv"
# Load into DataFrame
raw_data = pd.read_csv(csv_file)

# Show first 5 rows
raw_data.head()

##EDA

In [ ]:
raw_data.info()
raw_data.isnull().sum()

In [ ]:
raw_data["type"].value_counts().plot(kind="bar", title = "Transaction Types")
plt.xlabel("Transaction Types")
plt.ylabel("Count")
plt.show()

In [ ]:
fraud_by_type = raw_data.groupby("type")["isFraud"].mean().sort_values(ascending = False)
fraud_by_type.plot(kind = "bar", title = "Fraud Rate By Type", color = "salmon")
plt.ylabel("Count")
plt.show()

In [ ]:
raw_data.drop(columns="step", inplace=True)

##Feature Engineering and Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize, StandardScaler, OneHotEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [ ]:
df_model = raw_data.drop(["nameOrig", "nameDest", "isFlaggedFraud"], axis = 1)

In [ ]:
X = df_model.drop(["isFraud"], axis = 1)
y = df_model["isFraud"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

In [ ]:
#Split into numberical and categorical features
numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()  
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

In [ ]:
# Scale the numeric features
numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])

# One-hot encode the categoricals 
categorical_transformer = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))])

In [ ]:
#Combining the transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight="balanced", max_iter=1000))
])

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
confusion_matrix(y_test, y_pred)

In [ ]:
pipeline.score(X_test, y_test)*100

In [ ]:
import joblib

joblib.dump(pipeline, "fraud_detection_model.pkl")